# Holdout (2024-01-02 to 2026-08-31)

Run once, after everything in `03_evaluation.ipynb` is final.
Nothing may be changed in response to what this shows.

Success criterion 3: the holdout mean IC has the same sign as development, with |NW t| >= 2.

In [ ]:
import numpy as np
import pandas as pd

from shortvol import evaluate as ev, features, finra, prices

pd.set_option("display.width", 160)
pd.set_option("display.float_format", "{:.4f}".format)

panel = pd.read_parquet(prices.ROOT / "data" / "processed" / "panel.parquet")
days = finra.trading_days("2016-01-01", "2026-08-31")
V0 = features.estimate_v0(features.add_all(panel, days))["v0"]  # 2016-2019 short volume only
df = features.add_all(panel, days, v0=V0)

SIGNALS = ["svr5", "svr1", "svr_abnormal", "svr5_shrunk"]
CONTROLS = ["ret1", "ret5", "log_adv20", "abnormal_volume"]
TARGETS = ["fwd_oc", "fwd_cc"]
for s in SIGNALS:
    df[s + "_n"] = ev.neutralize(df, s, CONTROLS)
df["size_tercile"] = df.groupby("date").log_adv20.transform(
    lambda x: pd.qcut(x.rank(method="first"), 3, labels=["small", "mid", "large"]))
df["offex_tercile"] = df.groupby("date").off_exchange_share.transform(
    lambda x: pd.qcut(x.rank(method="first"), 3, labels=["low", "mid", "high"]))

In [ ]:
d = df[df.date >= "2024-01-02"]
print(f"holdout stock-days: {len(d):,}, days: {d.date.nunique()}")

In [ ]:
rows = []
for s in SIGNALS:
    for t in TARGETS:
        for kind, col in [("raw", s), ("neutralized", s + "_n")]:
            rows.append({"signal": s, "target": t, "version": kind, **ev.summarize_ic(ev.daily_ic(d, col, t))})
headline = pd.DataFrame(rows).set_index(["signal", "target", "version"])
headline

In [ ]:
dev = ev.summarize_ic(ev.daily_ic(df[df.date <= "2023-12-29"], "svr5_n", "fwd_oc"))
hold = headline.loc[("svr5", "fwd_oc", "neutralized")]
c3 = np.sign(hold.mean_ic) == np.sign(dev["mean_ic"]) and abs(hold.nw_t) >= 2
print(f"development mean IC {dev['mean_ic']:.4f}; holdout mean IC {hold.mean_ic:.4f}, NW t {hold.nw_t:.2f}")
print(f"3. {'PASS' if c3 else 'FAIL'}")

In [ ]:
pd.concat({
    "size": ev.ic_by_group(d, "svr5_n", "fwd_oc", "size_tercile"),
    "off-exchange share": ev.ic_by_group(d, "svr5_n", "fwd_oc", "offex_tercile"),
})